# Single-year vs multi-year ICESat-2 training: which AGBD model is better, why, and what it does to the boreal stock

**Question.** v3.1 AGBD maps can come from rF models trained on ICESat-2 ATL08 30 m observations from **one year** (the target year), or from a **multi-year** pool (2019–2024, each observation paired with HLS from its own year: "multi-year correspondence"). Which gives the better map, and how much does the choice change the estimates and the total boreal AGB stock?

| § | Evidence | What it asks | Role in the verdict |
|---|---|---|---|
| 2 | **A. Training data** | How many obs does each model learn from, how evenly do they cover the tile, and how does single-year availability vary by year? | Supporting (explains *why*) |
| 3 | **B. How the maps differ** | Where, and in which AGBD range, does the multi-year map raise or lower the estimate? | Descriptive (*where it matters*) |
| 4 | **C. Accuracy vs field plots** | Against the same screened plots, which map is closer, in which year, and is each change in the right direction? | **Primary** (*who is right*) |
| 5 | **D. Temporal stability** | In mature, undisturbed stands, which time series has less year-to-year noise? | Supporting |
| 6 | **E. Boreal AGB stock** | How much does the choice change the total stock in the boreal biome, where does the difference come from, and which total do the field-plot biases favour? | Effect size |

All differences are **multi − single** with 95 % bootstrap confidence intervals. A criterion counts as a win only when its CI excludes zero. **§7 gives the scorecard and verdict.** Design decisions are recorded in `model_strategy_comparison_LOG.md`.

**Years.** Every section runs for each year that has a single-year tile index in `SINGLE_TINDEX` (§0). ICESat-2 launched in September 2018, so single-year models are only possible for 2019–2025. Multi-year models also map 2016–2018 from the 2019–2024 pool paired with each year's HLS, which no single-year model can do.

*Why not the L-function.* ICESat-2 samples along ground tracks, so any ICESat-2 training set is strongly clustered by construction. Ripley's L(r) then mostly reflects track geometry and sample size, and cannot rank two training sets. The training data are instead summarised with directly interpretable metrics, and the maps are judged against field plots.

In [ ]:
suppressPackageStartupMessages({
    library(sf)
    library(terra)
    library(dplyr)
    library(tidyr)
    library(purrr)
    library(readr)
    library(stringr)
    library(ggplot2)
    library(patchwork)
    library(scales)
})
if (packageVersion("ggplot2") >= "3.5.0" && packageVersion("patchwork") < "1.2.0")
    stop("patchwork ", packageVersion("patchwork"), " cannot combine ggplot2 >= 3.5 plots. ",
         "Install patchwork 1.2.0 (remotes::install_version('patchwork', '1.2.0')) and restart the kernel.")
sf_use_s2(FALSE)
options(dplyr.summarise.inform = FALSE, width = 200, warn = 1)
set.seed(20261012)

## 0. Settings
`SINGLE_TINDEX` defines the comparison: add one entry per year as single-year runs become available (tiles in `../tiles/single_year_run_tiles.csv`). Run them with the multi-year run's version and settings (v3.1.0, `niter = 30`, ATL08 v6) so only the training years differ.

In [ ]:
## [config]
PROJ <- if (dir.exists("/projects")) "/projects" else "/home/jovyan"     # MAAP ADE: /projects; MAAP Hub: /home/jovyan
BASE <- "s3://maap-ops-workspace/shared/montesano/DPS_tile_lists/BOREAL_MAP"

# Single-year runs: one tile index per year. 2020 is the full-domain v3.1 run; add new single-year runs here.
SINGLE_TINDEX <- c(
    "2020" = file.path(BASE, "v3.1.0/AGB_H30_2020/full_run/AGB_tindex_master.csv")
    # "2021" = file.path(BASE, "v3.1.0/AGB_H30_2021/<single-year run identifier>/AGB_tindex_master.csv"),
)
MULTI_TINDEX <- function(year) file.path(BASE, sprintf("v3.1.0/AGB_H30_%d/full_run_niter30_multiyear_atl08v6/AGB_tindex_master.csv", year))
YEARS <- sort(as.integer(names(SINGLE_TINDEX)))

STRAT_LABELS <- c(single = "Single-year ICESat-2", multi = "Multi-year ICESat-2 (2019-2024)")
CONFOUNDS <- c("2020 single-year run: niter differs from the multi-year run (niter = 30) and it was run a month earlier; check both used the same mapBoreal commit")

# Tiles (the gpkg has no CRS; AEA is the map CRS) and the boreal biome
BOREAL_TILES_FN <- file.path(PROJ, "shared-buckets/montesano/databank/boreal_tiles_v004_model_ready.gpkg")
AEA <- st_crs("+proj=aea +lat_0=40 +lon_0=180 +lat_1=50 +lat_2=70 +x_0=0 +y_0=0 +ellps=GRS80 +towgs84=0,0,0,0,0,0,0 +units=m +no_defs")
WWF_ECO_FN   <- file.path(PROJ, "my-public-bucket/databank/wwf_terr_ecos.gpkg")
BOREAL_BIOME <- 6      # WWF BIOME code for "Boreal Forests/Taiga" (8 is "Temperate Grasslands, Savannas & Shrublands")

# §2-3 tile sample: the stratified + earlier-example tiles of the single-year run list, else a stratified draw
TILE_LIST_FN        <- "../tiles/single_year_run_tiles.csv"
LAT_BREAKS          <- seq(45, 75, by = 5)
N_TILES_PER_STRATUM <- 4                # used only without TILE_LIST_FN
N_PIX_PER_TILE      <- 2000             # §3 paired pixel sample per tile and year
AGBD_BINS           <- c(0, 10, 25, 50, 75, 100, 150, Inf)     # Mg/ha, for "by AGBD range" summaries

# §2 training-data metrics
PEAK_SEASON_DOY <- c(182, 243)          # Jul 1 - Aug 31
COVER_CELL_M    <- 3000                 # "share of tile covered" = share of 3 km cells with >= 1 training obs

# §4 field plots: per-plot screening table from maap_tools/notebooks/validation_v3.1_height_agbd.ipynb
VAL_DIR        <- file.path(PROJ, "my-private-bucket/validation_v3.1")
VAL_PLOTS_FN   <- Sys.getenv("CMP_VAL_PLOTS_FN")   # default: the latest validation run
if (VAL_PLOTS_FN == "") VAL_PLOTS_FN <- tail(c(NA, sort(Sys.glob(file.path(VAL_DIR, "*", "validation_plots_v3.1_screened.gpkg")))), 1)
NONFOREST_AGBD <- 5                     # Mg/ha; same threshold as the validation notebook
MATURE_AGE     <- 60                    # yr; §5 uses mature, undisturbed stands
CHANGE_TOL     <- 2                     # Mg/ha; smaller multi - single changes count as "little change" (§4)
MIN_PLOTS_BIN  <- 10                    # §6.3: minimum plots per map-AGBD bin to estimate a bias

# §5 temporal stability: single-year AGBD series once >= 4 single-year years exist; until then the v3.0 height
# series (2019-2024), which mixes model versions and is labelled as such
RUN_STABILITY <- TRUE
STABILITY <- if (length(YEARS) >= 4) {
    list(variable = "AGBD [Mg/ha]", years = YEARS, plot_excl = "excl_agbd", max_tiles = 80,
         tindex = list(single = function(y) SINGLE_TINDEX[[as.character(y)]], multi = MULTI_TINDEX),
         caveat = "same model version; only the training years differ")
} else {
    list(variable = "Vegetation height [m]", years = 2019:2024, plot_excl = "excl_ht", max_tiles = 80,
         tindex = list(single = function(y) file.path(BASE, sprintf("v3.0.0/Ht_H30_%d/full_run/HT_tindex_master.csv", y)),
                       multi  = function(y) file.path(BASE, sprintf("v3.1.0/Ht_H30_%d/full_run_niter30_multiyear_atl08v6/HT_tindex_master.csv", y))),
         caveat = "single-year series is v3.0 height, multi-year series is v3.1 (version differs as well as training years)")
}

# §6 stock: per-tile, per-ecoregion AGB totals by model iteration (*boreal_eco.csv). The stability notebook's
# combined table is used when it holds the run; otherwise each tile's file is read.
STOCK_TABLE_FN <- file.path(PROJ, "my-public-bucket/boreal_AGBD_trends/tables/boreal_eco_all_years_df.csv")

N_BOOT        <- 1000
N_CORES       <- 1                      # >1 uses parallel::mclapply over tiles
OUT_DIR       <- file.path(PROJ, "my-private-bucket/model_strategy_comparison", format(Sys.Date(), "%Y%m%d"))
CACHE_DIR     <- file.path(PROJ, "my-private-bucket/model_strategy_comparison/cache")   # per-tile results; reused across days
REFRESH_CACHE <- FALSE
dir.create(OUT_DIR, recursive = TRUE, showWarnings = FALSE)
dir.create(CACHE_DIR, recursive = TRUE, showWarnings = FALSE)
cat("Years compared:", YEARS, "| project root:", PROJ, "\n")

## Data access
All S3 reads go through these functions (tile index, run folder listing, candidate obs, training obs, map COG, stats CSV).

In [ ]:
## [data-access]
suppressPackageStartupMessages({
    library(aws.s3)
    library(arrow)
    library(sfarrow)
})
Sys.setenv(AWS_DEFAULT_REGION = "us-west-2")

# On MAAP, aws.s3 no longer finds the workspace's AWS credentials by itself (403 Access Denied), but the AWS CLI does.
# Copy the CLI's current credentials into this R session's environment, where aws.s3, arrow and GDAL all look for them.
# They are temporary, so they are refreshed every `max_age_min` minutes.
.s3_auth_time <- NULL
s3_auth <- function(max_age_min = 30) {
    if (!is.null(.s3_auth_time) && difftime(Sys.time(), .s3_auth_time, units = "mins") < max_age_min) return(invisible())
    out <- tryCatch(suppressWarnings(system2("aws", c("configure", "export-credentials", "--format", "env-no-export"),
                                             stdout = TRUE, stderr = FALSE)),
                    error = function(e) character(0))
    kv <- regmatches(out, regexec("^(AWS_[A-Z_]+)=(.*)$", out))
    kv <- kv[lengths(kv) == 3]
    if (!length(kv)) stop("Could not get AWS credentials from the AWS CLI. In a terminal, check that ",
                          "`aws configure export-credentials --format env-no-export` works (AWS CLI v2.9 or later).")
    do.call(Sys.setenv, setNames(lapply(kv, `[`, 3), vapply(kv, `[`, "", 2)))
    .s3_auth_time <<- Sys.time()
    invisible()
}
s3_auth()

to_vsis3 <- function(p) sub("^s3://", "/vsis3/", p)

# One row per tile; when a tile was re-run, keep the latest output (run timestamps are in the path)
read_tindex <- function(fn) {
    s3_auth()
    aws.s3::s3read_using(read.csv, object = fn) %>%
        select(tile_num, s3_path) %>%
        arrange(tile_num, desc(s3_path)) %>%
        distinct(tile_num, .keep_all = TRUE)
}

# All files in the DPS output folder of a tile's map COG
list_run_dir <- function(cog_path) {
    s3_auth()
    dir    <- dirname(sub("^s3://", "", cog_path))
    bucket <- sub("/.*", "", dir)
    prefix <- paste0(sub("^[^/]+/", "", dir), "/")
    keys   <- vapply(aws.s3::get_bucket(bucket, prefix = prefix, max = Inf), function(x) x$Key, "")
    paste0("s3://", bucket, "/", keys)
}

# Filtered ICESat-2 obs from the 3x3 tile window (merge_neighbors geoparquet; stored without a CRS)
read_candidates <- function(fn) {
    s3_auth()
    x <- sfarrow::st_read_parquet(fn)
    st_crs(x) <- AEA
    x %>% mutate(RH_98 = h_canopy) %>% select(any_of(c("RH_98", "y", "doy")))
}

# Obs actually used to fit the tile's rF model
read_training <- function(fn) {
    s3_auth()
    arrow::read_parquet(fn) %>%
        as.data.frame() %>%
        select(any_of(c("lon", "lat", "RH_98", "AGB", "y", "doy"))) %>%
        st_as_sf(coords = c("lon", "lat"), crs = 4326) %>%
        st_transform(AEA)
}

read_map <- function(cog_path) {
    s3_auth()
    terra::rast(to_vsis3(cog_path))
}

read_stats_csv <- function(fn) {
    s3_auth()
    aws.s3::s3read_using(read.csv, object = fn)
}

In [ ]:
STRAT_LEVELS <- c("single", "multi")
STRAT_COLORS <- c(single = "#d95f02", multi = "#1b9e77")
WINNER_COLORS <- c(single = STRAT_COLORS[["single"]], multi = STRAT_COLORS[["multi"]],
                   "no clear difference" = "grey70", "not tested" = "grey92", descriptive = "white")
CHANGE_COLORS <- c("raised, closer" = "#1b7837", "lowered, closer" = "#7fbf7b", "little change" = "grey85",
                   "lowered, further" = "#c2a5cf", "raised, further" = "#762a83")
scale_strategy <- function(...) scale_colour_manual(values = STRAT_COLORS, labels = STRAT_LABELS, name = NULL,
                                                    aesthetics = c("colour", "fill"), ...)
theme_set(theme_bw(base_size = 11) +
          theme(legend.position = "bottom", panel.grid.minor = element_blank(),
                plot.title = element_text(face = "bold"), plot.subtitle = element_text(colour = "grey30")))
agbd_bin <- function(x) cut(x, AGBD_BINS, right = FALSE, include.lowest = TRUE,
                            labels = paste0(head(AGBD_BINS, -1), ifelse(is.finite(AGBD_BINS[-1]), paste0("-", AGBD_BINS[-1]), "+")))

# Per-tile results are cached so re-runs only fetch what is new
cached <- function(name, expr) {
    fn <- file.path(CACHE_DIR, paste0(name, ".rds"))
    if (!REFRESH_CACHE && file.exists(fn)) return(readRDS(fn))
    val <- expr
    saveRDS(val, fn)
    val
}

# Apply f to each element of x; a failing element is reported and skipped
map_tiles <- function(x, f, label) {
    g <- function(k) tryCatch(f(k), error = function(e) {
        message(sprintf("  %s skipped (%s): %s", paste(k, collapse = "/"), label, conditionMessage(e))); NULL })
    out <- if (N_CORES > 1) parallel::mclapply(x, g, mc.cores = N_CORES) else lapply(x, g)
    message(sprintf("%s: %d of %d done", label, sum(!vapply(out, is.null, TRUE)), length(x)))
    out
}
# Run f(year, tile) over a table of (year, tile_num) tasks
map_tasks <- function(tasks, f, label) {
    g <- function(i) tryCatch(f(tasks$year[i], tasks$tile_num[i]), error = function(e) {
        message(sprintf("  %d / tile %s skipped (%s): %s", tasks$year[i], tasks$tile_num[i], label, conditionMessage(e))); NULL })
    out <- if (N_CORES > 1) parallel::mclapply(seq_len(nrow(tasks)), g, mc.cores = N_CORES) else lapply(seq_len(nrow(tasks)), g)
    message(sprintf("%s: %d of %d done", label, sum(!vapply(out, is.null, TRUE)), nrow(tasks)))
    out
}

cog_of <- function(tile, strategy, year) {
    b <- BOTH[[as.character(year)]]
    b[[paste0("s3_path_", strategy)]][b$tile_num == tile]
}

extract_at_points <- function(cog_path, pts) {
    r <- read_map(cog_path)
    v <- terra::extract(r, terra::vect(st_transform(pts, st_crs(terra::crs(r)))))
    unname(as.matrix(v[, -1, drop = FALSE]))       # drop the ID column; band 1 = mean, band 2 = sd
}

rmsd <- function(p, r) sqrt(mean((p - r)^2, na.rm = TRUE))

# Cluster bootstrap: resample whole clusters (tiles, or plot groups) with replacement
cluster_boot <- function(d, stat_fn, cluster_col = "cluster", n_boot = N_BOOT, min_clusters = 5) {
    est    <- stat_fn(d)
    groups <- split(seq_len(nrow(d)), d[[cluster_col]])
    res <- tibble(metric = names(est), estimate = unname(est), lo = NA_real_, hi = NA_real_,
                  n = nrow(d), n_clusters = length(groups))
    if (length(groups) < min_clusters) return(res)
    boots <- replicate(n_boot, stat_fn(d[unlist(groups[sample.int(length(groups), replace = TRUE)], use.names = FALSE), , drop = FALSE]))
    boots <- matrix(boots, nrow = length(est))
    res %>% mutate(lo = apply(boots, 1, quantile, 0.025, na.rm = TRUE),
                   hi = apply(boots, 1, quantile, 0.975, na.rm = TRUE))
}

# Winner of a (multi - single) difference, given which direction is better
call_winner <- function(lo, hi, better) {
    mapply(function(lo, hi, better) {
        if (is.na(better)) return("descriptive")
        if (is.na(lo) || is.na(hi)) return("not tested")
        if (lo > 0) return(if (better == "higher") "multi" else "single")
        if (hi < 0) return(if (better == "higher") "single" else "multi")
        "no clear difference"
    }, lo, hi, better, USE.NAMES = FALSE)
}

fmt_ci <- function(est, lo, hi, digits = 2) {
    ifelse(is.na(lo), formatC(est, format = "f", digits = digits),
           sprintf("%s [%s, %s]", formatC(est, format = "f", digits = digits),
                   formatC(lo, format = "f", digits = digits), formatC(hi, format = "f", digits = digits)))
}
winner_phrase <- function(w) dplyr::case_when(w == "multi" ~ "multi-year better", w == "single" ~ "single-year better",
                                              TRUE ~ w)
takeaway <- function(...) cat("\nTAKEAWAY:", ..., "\n")

## 1. Years and tiles compared
For each year, only tiles mapped by **both** runs are used. §2–3 use a fixed sample of tiles (the stratified and earlier-example tiles of the single-year run list, so the same tiles appear in every year). §4 uses every tile holding a screened field plot. §6 uses every tile with stock totals from both runs.

In [ ]:
TINDEX <- list(single = map(SINGLE_TINDEX, read_tindex),
               multi  = map(set_names(as.character(YEARS)), ~ read_tindex(MULTI_TINDEX(as.integer(.x)))))
BOTH <- map(set_names(as.character(YEARS)), function(y)
    inner_join(TINDEX$single[[y]], TINDEX$multi[[y]], by = "tile_num", suffix = c("_single", "_multi")))

boreal_tiles <- st_read(BOREAL_TILES_FN, quiet = TRUE)
st_crs(boreal_tiles) <- AEA
cen <- st_coordinates(st_transform(st_centroid(st_geometry(boreal_tiles)), 4326))
boreal_tiles <- boreal_tiles %>%
    mutate(lon = cen[, 1], lat = cen[, 2],
           continent = ifelse(lon > -170 & lon < -30, "North America", "Eurasia"),
           lat_band  = cut(lat, LAT_BREAKS, right = FALSE,
                           labels = paste0(head(LAT_BREAKS, -1), "-", LAT_BREAKS[-1], "°N")))
tile_info <- boreal_tiles %>% st_drop_geometry() %>% select(tile_num, lon, lat, continent, lat_band)

# Boreal biome polygons (pixels and plots are flagged in/out of the biome) and the ecoregion -> biome lookup
wwf <- st_read(WWF_ECO_FN, quiet = TRUE)
eco_biome <- wwf %>% st_drop_geometry() %>% distinct(ECO_ID, BIOME)
boreal_biome <- wwf %>% filter(BIOME == BOREAL_BIOME) %>% st_make_valid() %>% st_transform(AEA) %>% st_geometry()
in_boreal <- function(pts) lengths(st_intersects(st_transform(pts, AEA), boreal_biome)) > 0

all_common <- unique(unlist(map(BOTH, "tile_num")))
SAMPLE_TILES <- if (file.exists(TILE_LIST_FN)) {
    read_csv(TILE_LIST_FN, show_col_types = FALSE) %>% filter(reason != "field plots") %>% pull(tile_num)
} else {
    tile_info %>% filter(tile_num %in% all_common, !is.na(lat_band)) %>%
        group_by(continent, lat_band) %>% slice_sample(n = N_TILES_PER_STRATUM) %>% ungroup() %>% pull(tile_num)
}
SAMPLE_TILES <- sort(intersect(SAMPLE_TILES, all_common))

# §3 bins pixels by the multi-year map of another year (independent pixel noise); see the note in §3
BIN_YEAR <- function(y) if (y - 1 >= 2016) y - 1 else y + 1
BIN_TINDEX <- map(set_names(as.character(YEARS)), ~ read_tindex(MULTI_TINDEX(BIN_YEAR(as.integer(.x)))))
SAMPLE_TASKS <- map_dfr(names(BOTH), function(y) tibble(year = as.integer(y), tile_num = intersect(SAMPLE_TILES, BOTH[[y]]$tile_num)))

print(tibble(year = YEARS,
             tiles_single = map_int(TINDEX$single, nrow), tiles_multi = map_int(TINDEX$multi, nrow),
             tiles_both = map_int(BOTH, nrow), sampled = as.integer(table(factor(SAMPLE_TASKS$year, YEARS)))))

options(repr.plot.width = 10, repr.plot.height = 5, repr.plot.res = 150)
ggplot() +
    geom_sf(data = boreal_biome, fill = "grey92", colour = NA) +
    geom_sf(data = boreal_tiles %>% filter(tile_num %in% all_common), fill = "grey70", colour = NA) +
    geom_sf(data = boreal_tiles %>% filter(tile_num %in% SAMPLE_TILES), fill = "black", colour = NA) +
    labs(title = "Tiles compared", subtitle = sprintf("light grey: boreal biome; grey: mapped by both runs in >= 1 year (%d); black: sampled for §2-3 (%d)",
                                                     length(all_common), length(SAMPLE_TILES)))

## 2. Evidence A: what each model learns from
For each sampled tile, year and strategy, from the `*_train.parquet` (obs used to fit the rF model) and the `*merge_neighbors*.parquet` (all filtered candidate obs in the 3×3 tile window):

| Metric | Why it matters | Better |
|---|---|---|
| Training obs in tile | More obs → more stable rF fits and lower prediction variance | higher |
| Share of 3 km cells with ≥ 1 training obs | Even spatial coverage → fewer areas predicted far from any training data | higher |
| Median distance to nearest training obs | Same, as a distance (computed on a 1 km lattice over the tile) | lower |
| % obs in Jul–Aug, 95th pct training height | Timing and height range of the training set | descriptive |

Training obs carry no acquisition date, so each is matched to its candidate obs (nearest within 15 m and within 0.5 m in RH98). `pct_dated` reports how many matched.

In [ ]:
attach_obs_dates <- function(train, cand, max_dist = 15, max_dh = 0.5) {
    if (all(c("y", "doy") %in% names(train)) && any(!is.na(train$doy))) return(mutate(train, dated = !is.na(doy)))
    train <- train %>% mutate(y = NA_real_, doy = NA_real_, dated = FALSE)
    if (is.null(cand) || nrow(cand) == 0) return(train)
    i  <- st_nearest_feature(train, cand)
    d  <- as.numeric(st_distance(train, cand[i, ], by_element = TRUE))
    ok <- d <= max_dist & abs(train$RH_98 - cand$RH_98[i]) <= max_dh
    train$y[ok]   <- cand$y[i][ok]
    train$doy[ok] <- cand$doy[i][ok]
    train$dated   <- ok
    train
}

coverage_metrics <- function(pts, focal, cell = COVER_CELL_M, lattice = 1000) {
    grid <- st_make_grid(focal, cellsize = cell)
    grid <- grid[lengths(st_intersects(grid, focal)) > 0]
    lat_pts <- st_make_grid(focal, cellsize = lattice, what = "centers")
    d <- as.numeric(st_distance(lat_pts, pts[st_nearest_feature(lat_pts, pts), ], by_element = TRUE)) / 1000
    c(frac_cells = mean(lengths(st_intersects(grid, pts)) > 0),
      med_dist_km = median(d), p90_dist_km = unname(quantile(d, 0.9)))
}

summarise_training_tile <- function(year, t) {
    focal <- st_geometry(boreal_tiles[boreal_tiles$tile_num == t, ])
    res <- lapply(STRAT_LEVELS, function(s) {
        files   <- list_run_dir(cog_of(t, s, year))
        f_train <- grep("_train\\.parquet$", files, value = TRUE)[1]
        f_cand  <- grep("merge_neighbors.*\\.parquet$", files, value = TRUE)[1]
        if (is.na(f_train)) stop("no *_train.parquet for ", s)
        cand  <- if (!is.na(f_cand)) read_candidates(f_cand) else NULL
        train <- attach_obs_dates(read_training(f_train), cand)
        cov   <- coverage_metrics(train, focal)
        in_tile <- lengths(st_intersects(train, focal)) > 0
        tr <- st_drop_geometry(train) %>% mutate(in_tile = in_tile)
        dated <- tr %>% filter(dated)
        list(
            summary = tibble(
                year = year, tile_num = t, strategy = s,
                n_train = nrow(tr), n_train_tile = sum(in_tile),
                n_candidates = if (is.null(cand)) NA_integer_ else nrow(cand),
                frac_cells = cov[["frac_cells"]], med_dist_km = cov[["med_dist_km"]], p90_dist_km = cov[["p90_dist_km"]],
                pct_dated = 100 * mean(tr$dated),
                n_years = n_distinct(dated$y),
                pct_target_year = 100 * mean(dated$y == year),
                pct_peak_season = 100 * mean(dated$doy >= PEAK_SEASON_DOY[1] & dated$doy <= PEAK_SEASON_DOY[2]),
                rh98_p50 = median(tr$RH_98), rh98_p95 = unname(quantile(tr$RH_98, 0.95)),
                agb_p95 = if ("AGB" %in% names(tr)) unname(quantile(tr$AGB, 0.95, na.rm = TRUE)) else NA_real_),
            obs = tr %>% filter(in_tile) %>% slice_sample(n = 3000) %>%
                transmute(year = year, tile_num = t, strategy = s, RH_98, y, doy),
            # obs per ICESat-2 year in the tile: what a single-year model for that year would have had to work with
            year_counts = if (s == "multi") tr %>% filter(in_tile, dated) %>% count(icesat2_year = y) %>%
                              mutate(year = year, tile_num = t) else NULL)
    })
    list(summary = bind_rows(map(res, "summary")), obs = bind_rows(map(res, "obs")),
         year_counts = bind_rows(map(res, "year_counts")))
}

train_res <- map_tasks(SAMPLE_TASKS, function(y, t) cached(paste0("train_", y, "_", t), summarise_training_tile(y, t)), "training data")
train_sum   <- bind_rows(map(compact(train_res), "summary")) %>% left_join(tile_info, by = "tile_num")
train_obs   <- bind_rows(map(compact(train_res), "obs"))
year_counts <- bind_rows(map(compact(train_res), "year_counts")) %>% left_join(tile_info, by = "tile_num")
train_sum %>% group_by(year, strategy) %>%
    summarise(tiles = n(), across(c(n_train_tile, frac_cells, med_dist_km, pct_dated, pct_peak_season, rh98_p95), ~ round(median(.x, na.rm = TRUE), 2)))

In [ ]:
TRAIN_METRICS <- tribble(
    ~metric,           ~label,                                         ~better,  ~log2ratio,
    "n_train_tile",    "Training obs in tile",                         "higher", TRUE,
    "frac_cells",      "Share of 3-km cells with >= 1 training obs",   "higher", FALSE,
    "med_dist_km",     "Median distance to nearest training obs (km)", "lower",  FALSE,
    "pct_peak_season", "% training obs in Jul-Aug",                    NA,       FALSE,
    "rh98_p95",        "95th pct training height (m)",                 NA,       FALSE)

# Paired by tile and year: difference (multi - single), or log2 ratio for counts; bootstrap over tiles
train_paired <- train_sum %>%
    select(year, tile_num, strategy, all_of(TRAIN_METRICS$metric)) %>%
    pivot_longer(-c(year, tile_num, strategy), names_to = "metric") %>%
    pivot_wider(names_from = strategy, values_from = value) %>%
    filter(!is.na(single), !is.na(multi)) %>%
    left_join(TRAIN_METRICS, by = "metric") %>%
    mutate(paired = ifelse(log2ratio, log2(pmax(multi, 1) / pmax(single, 1)), multi - single))

train_test_fn <- function(d) cluster_boot(d %>% mutate(cluster = tile_num), function(x) c(diff = mean(x$paired))) %>%
    select(-metric) %>%
    mutate(single = median(d$single), multi = median(d$multi), pct_tiles_multi_higher = 100 * mean(d$multi > d$single))
train_tests <- bind_rows(
    train_paired %>% group_by(metric) %>% group_modify(~ train_test_fn(.x)) %>% ungroup() %>% mutate(year = "all"),
    train_paired %>% group_by(metric, year) %>% group_modify(~ train_test_fn(.x)) %>% ungroup() %>% mutate(year = as.character(year))) %>%
    left_join(TRAIN_METRICS, by = "metric") %>%
    mutate(winner = call_winner(lo, hi, better))
print(train_tests %>% filter(year == "all") %>% select(label, single, multi, estimate, lo, hi, pct_tiles_multi_higher, winner))

options(repr.plot.width = 14, repr.plot.height = 4.5, repr.plot.res = 150)
pA1 <- ggplot(train_paired %>% filter(!is.na(better)) %>%
                  mutate(label = ifelse(log2ratio, paste(label, "(log2 multi/single)"), paste(label, "(multi - single)"))),
              aes(factor(year), paired)) +
    geom_hline(yintercept = 0) +
    geom_boxplot(outlier.size = 0.6, fill = STRAT_COLORS[["multi"]], alpha = 0.35) +
    facet_wrap(~ label, scales = "free_y", nrow = 1) +
    labs(x = "Target year", y = "Paired difference per tile", title = "A1. Training data: multi-year relative to single-year, per tile",
         subtitle = "Above zero: multi-year higher. The first two panels favour multi-year above zero; the third below zero")
pA1

**Reading A1.** Each box is the spread over tiles of the paired difference for one target year. If the multi-year advantage is real but year-dependent, the boxes move further from zero in years with poor ICESat-2 acquisitions.

In [ ]:
options(repr.plot.width = 14, repr.plot.height = 5, repr.plot.res = 150)
# A0: what a single-year model would have had, by ICESat-2 year, from the multi-year pool (any one target-year run)
yc <- year_counts %>% filter(year == year[1]) %>%
    complete(nesting(tile_num, lat_band, continent), icesat2_year = 2016:2025, fill = list(n = 0)) %>%
    group_by(continent, lat_band, icesat2_year) %>% summarise(median_obs = median(n), tiles = n()) %>% ungroup() %>%
    filter(!is.na(lat_band))
pA0 <- ggplot(yc, aes(factor(icesat2_year), lat_band, fill = median_obs)) +
    geom_tile(colour = "white") +
    geom_text(aes(label = ifelse(icesat2_year < 2019, "-", comma(median_obs))), size = 2.8) +
    facet_wrap(~ continent) +
    scale_fill_viridis_c(trans = "sqrt", name = "median\nobs / tile") +
    labs(x = "ICESat-2 acquisition year", y = NULL,
         title = "A0. Training obs a single-year model would have had, by year",
         subtitle = "Median per tile, from the dated obs in the multi-year training pool. 2016-2018: no ICESat-2 growing season") +
    theme(legend.position = "right")

obs_dated <- train_obs %>% filter(!is.na(doy))
pA2 <- ggplot(obs_dated, aes(x = doy, fill = strategy)) +
    geom_histogram(binwidth = 7, position = "identity", alpha = 0.55) +
    annotate("rect", xmin = PEAK_SEASON_DOY[1], xmax = PEAK_SEASON_DOY[2], ymin = -Inf, ymax = Inf, alpha = 0.08) +
    scale_x_continuous(breaks = c(152, 182, 213, 244, 274), labels = c("Jun 1", "Jul 1", "Aug 1", "Sep 1", "Oct 1")) +
    scale_strategy() +
    labs(x = "Acquisition date", y = "Training obs (sampled tiles)", title = "A2. When the training obs were acquired",
         subtitle = "Shaded: Jul-Aug peak season")
pA3 <- ggplot(train_obs, aes(x = RH_98, colour = strategy)) +
    stat_ecdf(linewidth = 0.9) +
    scale_x_continuous(limits = c(0, quantile(train_obs$RH_98, 0.995, na.rm = TRUE))) +
    scale_strategy() + guides(colour = "none") +      # legend shown once, in A2
    labs(x = "Training height RH98 (m)", y = "Cumulative share", title = "A3. Height range of training obs",
         subtitle = "A curve further right reaches taller stands")
print(pA0)
print(pA2 + pA3 + plot_layout(widths = c(1.6, 1)))

tr_n <- train_tests %>% filter(metric == "n_train_tile", year == "all")
tr_c <- train_tests %>% filter(metric == "frac_cells", year == "all")
yr_cv <- year_counts %>% filter(year == year[1]) %>%
    complete(tile_num, icesat2_year = 2019:2024, fill = list(n = 0)) %>% filter(icesat2_year %in% 2019:2024) %>%
    group_by(tile_num) %>% summarise(ratio = max(n) / pmax(min(n), 1))
takeaway(sprintf("over %s, multi-year training has %.1fx the obs per tile [%.1fx, %.1fx] (more in %.0f%% of tile-years) and covers %+.0f percentage points more of each tile's 3-km cells [%+.0f, %+.0f]. In the median tile, the best ICESat-2 year has %.1fx the obs of the worst, so single-year models vary strongly in data from year to year.",
                 paste(YEARS, collapse = ", "), 2^tr_n$estimate, 2^tr_n$lo, 2^tr_n$hi, tr_n$pct_tiles_multi_higher,
                 100 * tr_c$estimate, 100 * tr_c$lo, 100 * tr_c$hi, median(yr_cv$ratio)))

**Reading A0.** Each cell is the median number of training obs per tile that a model for that single ICESat-2 year would have had (the same filtering as the multi-year pool). Low cells are the years and latitudes where a single-year model is starved for data; the multi-year model uses all columns at once, for every target year including 2016–2018.

## 3. Evidence B: how the maps differ, and in which AGBD range
The same `N_PIX_PER_TILE` pixels (a regular grid) are read from both maps for each sampled tile and year. B1–B3 show *where* the strategy changes the map. B4 answers *in which AGBD range*, and B5 how much each range contributes to the overall difference (area share × mean difference). Only pixels inside the boreal biome are used in B4–B5. Whether the changes are improvements is §4.

**How pixels are binned.** Binning by either compared map, or by their mean, builds in a regression-to-the-mean artifact: when one map is noisier, its high-noise pixels fall into the high bins and make the other map look lower there (and the reverse in low bins), even if the true difference is the same everywhere. Pixels are therefore binned by a **reference AGBD that shares neither map's pixel noise**: the multi-year map of the previous year (`BIN_YEAR()`). Systematic model errors shared by multi-year maps of different years can still leave a small artifact for the multi-year side.

In [ ]:
compare_maps_tile <- function(year, t) {
    rs <- read_map(cog_of(t, "single", year))
    xy <- terra::spatSample(rs[[1]], size = N_PIX_PER_TILE, method = "regular", xy = TRUE, na.rm = FALSE)[, c("x", "y")]
    pts <- st_as_sf(xy, coords = c("x", "y"), crs = st_crs(terra::crs(rs)))
    vs <- extract_at_points(cog_of(t, "single", year), pts)
    vm <- extract_at_points(cog_of(t, "multi", year), pts)
    ref_path <- with(BIN_TINDEX[[as.character(year)]], s3_path[tile_num == t])
    vr <- if (length(ref_path)) extract_at_points(ref_path, pts)[, 1] else NA_real_
    tibble(year = year, tile_num = t, in_boreal = in_boreal(pts), agb_ref = vr,
           agb_single = vs[, 1], sd_single = if (ncol(vs) > 1) vs[, 2] else NA_real_,
           agb_multi  = vm[, 1], sd_multi  = if (ncol(vm) > 1) vm[, 2] else NA_real_) %>%
        filter(!is.na(agb_single), !is.na(agb_multi))
}

pix <- bind_rows(map_tasks(SAMPLE_TASKS, function(y, t) cached(paste0("pix2_", y, "_", t, "_", N_PIX_PER_TILE), compare_maps_tile(y, t)), "map comparison")) %>%
    mutate(diff = agb_multi - agb_single, bin = agbd_bin(agb_ref)) %>%
    left_join(tile_info, by = "tile_num")

tile_diff <- pix %>% group_by(year, tile_num, lat, lat_band, continent) %>%
    summarise(n = n(), mean_single = mean(agb_single), mean_multi = mean(agb_multi), mean_diff = mean(diff),
              rmsd_between = sqrt(mean(diff^2)), sd_single = mean(sd_single), sd_multi = mean(sd_multi)) %>%
    ungroup() %>%
    left_join(train_sum %>% select(year, tile_num, strategy, n_train_tile) %>%
                  pivot_wider(names_from = strategy, values_from = n_train_tile, names_prefix = "n_"), by = c("year", "tile_num")) %>%
    mutate(log2_n_ratio = log2(pmax(n_multi, 1) / pmax(n_single, 1)))

pix_b <- pix %>% filter(in_boreal) %>% mutate(cluster = tile_num)
if (mean(is.na(pix_b$bin)) > 0.2) message(sprintf("%.0f%% of boreal pixels lack a reference-year value and are left out of B4-B5", 100 * mean(is.na(pix_b$bin))))
map_tests <- bind_rows(
    cluster_boot(pix_b, function(d) c(mean_diff = mean(d$diff))),
    cluster_boot(pix_b %>% filter(!is.na(sd_single), !is.na(sd_multi)), function(d) c(sd_diff = mean(d$sd_multi - d$sd_single))))

# B4: by AGBD range (mean of the two maps), boreal pixels
by_bin <- pix_b %>%
    filter(!is.na(bin)) %>%
    group_by(bin) %>%
    group_modify(~ cluster_boot(.x, function(d) c(mean_diff = mean(d$diff))) %>% select(-metric) %>%
                     mutate(area_share = nrow(.x) / sum(!is.na(pix_b$bin)), q25 = quantile(.x$diff, 0.25), q75 = quantile(.x$diff, 0.75))) %>%
    ungroup() %>%
    mutate(contribution = area_share * estimate,                         # Mg/ha of the boreal-wide mean difference
           pct_of_total = 100 * contribution / sum(contribution))
map_tests
by_bin %>% select(bin, area_share, estimate, lo, hi, q25, q75, contribution, pct_of_total)

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 5, repr.plot.res = 150)
lim <- quantile(c(pix$agb_single, pix$agb_multi), 0.995)
pB1 <- ggplot(pix, aes(agb_single, agb_multi)) +
    geom_bin2d(bins = 70) +
    geom_abline(linetype = "dashed") +
    scale_fill_viridis_c(trans = "log10", name = "pixels") +
    coord_equal(xlim = c(0, lim), ylim = c(0, lim)) +
    labs(x = "Single-year AGBD (Mg/ha)", y = "Multi-year AGBD (Mg/ha)", title = "B1. Same pixels, two maps",
         subtitle = sprintf("%s pixels, %d tiles, %s", comma(nrow(pix)), n_distinct(pix$tile_num), paste(YEARS, collapse = ", ")))
pB2 <- ggplot(pix, aes(x = lat_band, y = diff, fill = continent)) +
    geom_hline(yintercept = 0) +
    geom_boxplot(outlier.shape = NA, width = 0.7) +
    coord_cartesian(ylim = quantile(pix$diff, c(0.02, 0.98))) +
    scale_fill_manual(values = c("North America" = "grey80", "Eurasia" = "grey50"), name = NULL) +
    labs(x = NULL, y = "Multi - single (Mg/ha)", title = "B2. Difference by latitude",
         subtitle = "Box: pixel differences within each band (whiskers trimmed)") +
    theme(axis.text.x = element_text(angle = 45, hjust = 1))
pB3 <- ggplot(tile_diff, aes(log2_n_ratio, rmsd_between)) +
    geom_point(aes(colour = lat)) +
    geom_smooth(method = "lm", formula = y ~ x, colour = "black", linewidth = 0.6) +
    scale_colour_viridis_c(name = "Latitude") +
    labs(x = "log2(multi / single training obs in tile)", y = "RMS difference between maps (Mg/ha)",
         title = "B3. Disagreement vs training-data gap", subtitle = "Each point is a tile-year")
print(pB1 + pB2 + pB3 + plot_layout(widths = c(1, 1.3, 1.1)))

pB4a <- ggplot(by_bin, aes(bin, estimate)) +
    geom_hline(yintercept = 0) +
    geom_linerange(aes(ymin = q25, ymax = q75), colour = "grey70", linewidth = 4) +
    geom_pointrange(aes(ymin = lo, ymax = hi), colour = "black") +
    labs(x = "Reference AGBD: multi-year map of the previous year (Mg/ha)", y = "Multi - single (Mg/ha)",
         title = "B4. Where the multi-year map is higher or lower",
         subtitle = "Point and bar: mean with 95% CI; grey: middle 50% of pixels. Boreal biome pixels")
pB4b <- ggplot(by_bin, aes(bin, contribution, fill = contribution > 0)) +
    geom_col(width = 0.7) +
    geom_hline(yintercept = 0) +
    geom_text(aes(label = sprintf("%.0f%% of area", 100 * area_share), vjust = ifelse(contribution > 0, -0.4, 1.3)), size = 3) +
    scale_fill_manual(values = c("TRUE" = STRAT_COLORS[["multi"]], "FALSE" = STRAT_COLORS[["single"]]), guide = "none") +
    labs(x = "Reference AGBD (Mg/ha)", y = "Contribution to mean difference (Mg/ha)",
         title = "B5. Which AGBD range drives the overall difference",
         subtitle = sprintf("Area share x mean difference; bars sum to the boreal-wide mean difference (%+.2f Mg/ha)", sum(by_bin$contribution)))
print(pB4a + pB4b)

md_cor <- tryCatch(with(tile_diff, suppressWarnings(cor.test(log2_n_ratio, rmsd_between, method = "spearman", exact = FALSE))),
                   error = function(e) list(estimate = NA_real_, p.value = NA_real_))
md_diff <- map_tests %>% filter(metric == "mean_diff")
top_up <- by_bin %>% slice_max(contribution, n = 1)
top_dn <- by_bin %>% slice_min(contribution, n = 1)
range_txt <- paste0(
    if (top_up$contribution > 0) sprintf("The largest upward contribution comes from %s Mg/ha stands (%+.2f Mg/ha)", top_up$bin, top_up$contribution)
    else "No AGBD range is raised",
    if (top_dn$contribution < 0) sprintf("; the largest downward from %s Mg/ha (%+.2f Mg/ha)", top_dn$bin, top_dn$contribution)
    else "; no range is lowered")
takeaway(sprintf("in the boreal biome, the multi-year map is %+.1f Mg/ha [%+.1f, %+.1f] relative to single-year on average. %s. %.0f%% of pixels differ by > 20 Mg/ha. Tiles where multi-year has relatively more training obs disagree %s (Spearman rho = %.2f, p = %.2f).",
                 md_diff$estimate, md_diff$lo, md_diff$hi, range_txt,
                 100 * mean(abs(pix$diff) > 20), ifelse(isTRUE(md_cor$estimate > 0), "more", "less"), md_cor$estimate, md_cor$p.value))

**Reading B.** B1: points on the dashed line are pixels where the strategy makes no difference. B2: differences growing toward the north mean the strategy matters most where the single-year set is thinnest. B3: a rising trend ties the map differences to the training-data differences. **B4–B5** (binned by reference AGBD, see above) are the key figures for the stock question: B4 shows whether the multi-year map raises or lowers each AGBD range (offsetting changes are visible here even when the overall mean difference is small), and B5 shows how much each range contributes to the overall difference once its area is taken into account.

The maps' own uncertainty bands (`sd_diff`) are descriptive only: the 2020 multi-year run used `niter = 30`, which changes the sd independently of training data.

## 4. Evidence C: accuracy against field plots (who is right)
Field plots come from the screened table written by `validation_v3.1_height_agbd.ipynb`. Kept: plots not excluded a priori for AGBD, with a stand-age class of *undisturbed* or *marginal*, or non-forest plots without a mapped age (the validation's default tier). Every year's pair of maps is read at every plot in a tile mapped by both runs that year.

**Design: paired.** Both maps are compared against the same reference value at the same plot. Most plots were measured years before the map year, so both maps carry the same reference-age error; it cancels in the paired difference.

- **C1–C3**: bias, RMSD and MAE per strategy, and paired differences (multi − single) with 95 % CIs from a cluster bootstrap over plot groups (source × group × field year), pooled over years and per year.
- **C4: direction of change.** At each plot, switching from single-year to multi-year either *raised* or *lowered* the estimate, and moved it *closer to* or *further from* the field value (changes smaller than `CHANGE_TOL` count as "little change"). Shown by field AGBD range: this is where the map improves or degrades, and in which direction.

In [ ]:
if (is.na(VAL_PLOTS_FN) || !file.exists(VAL_PLOTS_FN))
    stop("No screened field plots found. Run maap_tools/notebooks/validation_v3.1_height_agbd.ipynb first, or set CMP_VAL_PLOTS_FN.")
message("Field plots: ", VAL_PLOTS_FN)

val_all <- st_read(VAL_PLOTS_FN, quiet = TRUE) %>%
    mutate(across(any_of(c("excl_agbd", "excl_ht")), as.logical), age_class = as.character(age_class))
plots_tiled <- val_all %>%
    st_transform(AEA) %>%
    st_join(boreal_tiles %>% select(tile_num), join = st_intersects, left = FALSE) %>%
    distinct(uid, .keep_all = TRUE) %>%
    filter(tile_num %in% all_common) %>%
    mutate(in_boreal = in_boreal(.))

acc_plots <- plots_tiled %>%
    filter(!excl_agbd, !is.na(ref_agbd),
           age_class %in% c("undisturbed", "marginal") | (age_class == "no mapped age" & ref_agbd < NONFOREST_AGBD))

extract_plots_tile <- function(year, t, pts) {
    p <- pts[pts$tile_num == t, ]
    out <- tibble(uid = p$uid, year = year)
    for (s in STRAT_LEVELS) out[[paste0("pred_", s)]] <- extract_at_points(cog_of(t, s, year), p)[, 1]
    out
}
acc_tasks <- map_dfr(names(BOTH), function(y) tibble(year = as.integer(y), tile_num = intersect(unique(acc_plots$tile_num), BOTH[[y]]$tile_num)))
pred_plots <- bind_rows(map_tasks(acc_tasks, function(y, t)
    cached(paste0("plots_", y, "_", t, "_n", sum(acc_plots$tile_num == t)), extract_plots_tile(y, t, acc_plots)), "field plots"))

pairs <- acc_plots %>% st_drop_geometry() %>%
    inner_join(pred_plots, by = "uid") %>%
    filter(!is.na(pred_single), !is.na(pred_multi)) %>%
    mutate(cluster = paste(source, group_name, year_obs),
           err_single = pred_single - ref_agbd, err_multi = pred_multi - ref_agbd,
           ref_bin = agbd_bin(ref_agbd),
           delta = pred_multi - pred_single,
           change = case_when(abs(delta) < CHANGE_TOL ~ "little change",
                              delta > 0 & abs(err_multi) < abs(err_single) ~ "raised, closer",
                              delta > 0 ~ "raised, further",
                              abs(err_multi) < abs(err_single) ~ "lowered, closer",
                              TRUE ~ "lowered, further"),
           change = factor(change, names(CHANGE_COLORS)))
cat(sprintf("Paired plot-years: %d (%d plots, %d plot groups, %d tiles, years %s)\n", nrow(pairs), n_distinct(pairs$uid),
            n_distinct(pairs$cluster), n_distinct(pairs$tile_num), paste(sort(unique(pairs$year)), collapse = ", ")))
print(pairs %>% count(year, region) %>% pivot_wider(names_from = region, values_from = n))

In [ ]:
per_strategy <- function(d) c(bias_single = mean(d$err_single), bias_multi = mean(d$err_multi),
                              rmsd_single = rmsd(d$pred_single, d$ref_agbd), rmsd_multi = rmsd(d$pred_multi, d$ref_agbd),
                              mae_single = mean(abs(d$err_single)), mae_multi = mean(abs(d$err_multi)))
paired_stats <- function(d) c(d_rmsd    = rmsd(d$pred_multi, d$ref_agbd) - rmsd(d$pred_single, d$ref_agbd),
                              d_mae     = mean(abs(d$err_multi)) - mean(abs(d$err_single)),
                              d_absbias = abs(mean(d$err_multi)) - abs(mean(d$err_single)),
                              pct_multi_closer_minus50 = 100 * mean(abs(d$err_multi) < abs(d$err_single)) - 50)

HEADLINE <- "All years, boreal plots"
boreal_pairs <- pairs %>% filter(region != "Temperate")
acc_tests <- bind_rows(
    cluster_boot(boreal_pairs, paired_stats) %>% mutate(domain = HEADLINE),
    boreal_pairs %>% group_by(domain = paste("Year", year)) %>% group_modify(~ cluster_boot(.x, paired_stats)) %>% ungroup(),
    boreal_pairs %>% group_by(domain = paste("Region:", region)) %>% group_modify(~ cluster_boot(.x, paired_stats)) %>% ungroup(),
    if (any(pairs$region == "Temperate"))
        cluster_boot(pairs %>% filter(region == "Temperate"), paired_stats) %>% mutate(domain = "Temperate (not in headline)")) %>%
    mutate(better = ifelse(metric == "pct_multi_closer_minus50", "higher", "lower"),
           winner = call_winner(lo, hi, better))
acc_abs <- cluster_boot(boreal_pairs, per_strategy)
acc_abs_year <- boreal_pairs %>% group_by(year) %>% group_modify(~ cluster_boot(.x, per_strategy)) %>% ungroup()
print(acc_abs)
acc_tests %>% filter(metric == "d_rmsd") %>% select(domain, estimate, lo, hi, n, n_clusters, winner)

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 5.5, repr.plot.res = 150)
long_pairs <- boreal_pairs %>%
    select(uid, year, region, ref_agbd, single = pred_single, multi = pred_multi) %>%
    pivot_longer(c(single, multi), names_to = "strategy", values_to = "pred") %>%
    mutate(strategy = factor(strategy, STRAT_LEVELS))
lab <- long_pairs %>% group_by(strategy) %>%
    summarise(txt = sprintf("bias %+.1f | RMSD %.1f | MAE %.1f Mg/ha", mean(pred - ref_agbd), rmsd(pred, ref_agbd), mean(abs(pred - ref_agbd))))
lim <- quantile(c(long_pairs$pred, long_pairs$ref_agbd), 0.995, na.rm = TRUE)
pC1 <- ggplot(long_pairs, aes(ref_agbd, pred, colour = strategy)) +
    geom_abline(linetype = "dashed") +
    geom_point(alpha = 0.3, size = 0.8) +
    geom_text(data = lab, aes(x = 0, y = lim, label = txt), hjust = 0, vjust = 1, size = 3.2, colour = "black", inherit.aes = FALSE) +
    facet_wrap(~ strategy, labeller = as_labeller(STRAT_LABELS)) +
    coord_equal(xlim = c(0, lim), ylim = c(0, lim)) +
    scale_strategy() + guides(colour = "none") +
    labs(x = "Field AGBD (Mg/ha)", y = "Map AGBD (Mg/ha)", title = "C1. Both maps against the same field plots",
         subtitle = paste("All compared years:", paste(YEARS, collapse = ", ")))

forest <- acc_tests %>% filter(metric %in% c("d_rmsd", "d_absbias")) %>%
    mutate(metric = recode(metric, d_rmsd = "RMSD", d_absbias = "|bias|"),
           domain = factor(domain, rev(unique(domain))))
pC2 <- ggplot(forest, aes(estimate, domain, colour = winner)) +
    geom_vline(xintercept = 0) +
    geom_errorbar(aes(xmin = lo, xmax = hi), width = 0.25, orientation = "y") +
    geom_point(size = 2.3) +
    facet_wrap(~ metric, scales = "free_x") +
    scale_colour_manual(values = WINNER_COLORS, name = NULL) +
    labs(x = "Multi - single (Mg/ha); left of 0 = multi-year more accurate", y = NULL,
         title = "C2. Paired accuracy difference", subtitle = "Pooled, per year, and per region; 95% cluster-bootstrap CI")
print(pC1 + pC2 + plot_layout(widths = c(1.4, 1)))

by_ref <- boreal_pairs %>%
    group_by(ref_bin) %>%
    summarise(n = n(), single = mean(abs(err_single)), multi = mean(abs(err_multi))) %>%
    pivot_longer(c(single, multi), names_to = "strategy", values_to = "mae")
pC3 <- ggplot(by_ref, aes(ref_bin, mae, colour = strategy, group = strategy)) +
    geom_line(linewidth = 0.9) + geom_point(size = 2) +
    geom_text(data = by_ref %>% filter(strategy == "multi"), aes(y = 0, label = paste0("n=", n)), colour = "grey40", vjust = 0, size = 3) +
    scale_strategy() +
    labs(x = "Field AGBD (Mg/ha)", y = "Mean absolute error (Mg/ha)", title = "C3. Error by field AGBD range", subtitle = "Lower is better")
change_tab <- boreal_pairs %>% count(ref_bin, change) %>% group_by(ref_bin) %>% mutate(share = n / sum(n)) %>% ungroup()
net_tab <- boreal_pairs %>% group_by(ref_bin) %>%
    summarise(n = n(), mean_delta = mean(delta), net_abs_err_change = mean(abs(err_multi) - abs(err_single)),
              pct_closer = 100 * mean(abs(err_multi) < abs(err_single)))
pC4 <- ggplot(change_tab, aes(ref_bin, share, fill = change)) +
    geom_col(width = 0.75) +
    geom_text(data = net_tab, aes(x = ref_bin, y = 1.02, label = sprintf("%+.1f", net_abs_err_change)), inherit.aes = FALSE, vjust = 0, size = 3) +
    scale_y_continuous(labels = percent, expand = expansion(mult = c(0, 0.08))) +
    scale_fill_manual(values = CHANGE_COLORS, name = NULL) +
    labs(x = "Field AGBD (Mg/ha)", y = "Share of plot-years",
         title = "C4. Direction of change: single-year -> multi-year",
         subtitle = "Green: the change moved the estimate toward the field value. Top: mean change in |error| (Mg/ha; negative = better)") +
    theme(legend.position = "right")
print(pC3 + pC4 + plot_layout(widths = c(1, 1.5)))
net_tab

In [ ]:
d_r <- acc_tests %>% filter(domain == HEADLINE, metric == "d_rmsd")
d_c <- acc_tests %>% filter(domain == HEADLINE, metric == "pct_multi_closer_minus50")
yr  <- acc_tests %>% filter(metric == "d_rmsd", str_detect(domain, "^Year")) %>% arrange(estimate)
up_fix   <- net_tab %>% left_join(change_tab %>% filter(change == "raised, closer") %>% select(ref_bin, raised_closer = share), by = "ref_bin") %>%
    slice_max(raised_closer, n = 1, with_ties = FALSE)
takeaway(sprintf("against %d boreal plot-years (%d plot groups), RMSD single = %.1f, multi = %.1f Mg/ha; difference %s Mg/ha (%s). The multi-year map is closer at %.0f%% of plot-years [%.0f, %.0f]. %s Most common improvement by raising the estimate: %s Mg/ha plots (%.0f%% raised and closer).",
                 d_r$n, d_r$n_clusters, acc_abs$estimate[acc_abs$metric == "rmsd_single"], acc_abs$estimate[acc_abs$metric == "rmsd_multi"],
                 fmt_ci(d_r$estimate, d_r$lo, d_r$hi, 1), winner_phrase(d_r$winner), 50 + d_c$estimate, 50 + d_c$lo, 50 + d_c$hi,
                 if (nrow(yr) > 1) sprintf("By year, the multi-year advantage is largest in %s (%+.1f Mg/ha) and smallest in %s (%+.1f).",
                                           sub("Year ", "", yr$domain[1]), yr$estimate[1], sub("Year ", "", yr$domain[nrow(yr)]), yr$estimate[nrow(yr)]) else "",
                 if (nrow(up_fix)) as.character(up_fix$ref_bin) else "none", if (nrow(up_fix)) 100 * up_fix$raised_closer else 0))

**Reading C.** C2 is the decision figure: an interval wholly left of zero means the multi-year map is more accurate in that year or region. C3 and C4 show *where* along the biomass range the difference arises and *how*: in C4, a tall green "raised, closer" segment in high-biomass bins means the multi-year model fixes underestimation of dense forest; purple segments are changes that made things worse. The number above each bar is the net effect on |error| in that range.

Field plots are mostly pre-2016, plot-scale (314–707 m²) against 900 m² pixels, and clustered in groups. These affect both maps equally in the paired design, but they limit how small a difference can be detected.

## 5. Evidence D: temporal stability
In mature (age ≥ `MATURE_AGE`), undisturbed stands, real AGBD and height change little from year to year. Year-to-year variation in the map there is mostly noise, so the strategy with **less** variation is more trustworthy for trends. Per plot: the standard deviation across years and the mean absolute year-to-year change, paired by plot (multi − single) and bootstrapped over tiles.

- With single-year AGBD runs for ≥ 4 years, this section compares AGBD series of the same model version. Until then it falls back to height (single-year v3.0 vs multi-year v3.1), which is supporting evidence only.
- Multi-year models share their training pool across years, so they are smoother partly by construction. Lower noise is only a virtue if real change is still detected; the LVIS 2017→2019 change test in `validation_v3.1_lidar_LVIS_GLiHT.ipynb` checks that.

In [ ]:
stab_tests <- NULL
if (RUN_STABILITY) {
    STAB_TINDEX <- map(STABILITY$tindex, function(f) {
        bind_rows(lapply(STABILITY$years, function(y) tryCatch(read_tindex(f(y)) %>% mutate(year = y),
                                                               error = function(e) { message("  no tindex for ", y, ": ", conditionMessage(e)); NULL })))
    })
    if (any(vapply(STAB_TINDEX, nrow, 1L) == 0)) stop("no stability tile index found for one strategy; set RUN_STABILITY <- FALSE or fix STABILITY$tindex")
    stab_plots <- plots_tiled %>%
        filter(!.data[[STABILITY$plot_excl]], age_class == "undisturbed", !is.na(age_map), age_map >= MATURE_AGE)
    stab_tiles <- intersect(unique(stab_plots$tile_num), intersect(STAB_TINDEX$single$tile_num, STAB_TINDEX$multi$tile_num))
    if (length(stab_tiles) > STABILITY$max_tiles) stab_tiles <- sort(sample(stab_tiles, STABILITY$max_tiles))

    extract_series_tile <- function(t) {
        p <- stab_plots[stab_plots$tile_num == t, ]
        bind_rows(lapply(STRAT_LEVELS, function(s) {
            ti <- STAB_TINDEX[[s]] %>% filter(tile_num == t)
            bind_rows(lapply(seq_len(nrow(ti)), function(i)
                tibble(uid = p$uid, tile_num = t, strategy = s, year = ti$year[i],
                       value = extract_at_points(ti$s3_path[i], p)[, 1])))
        }))
    }
    stab_key <- paste0("series_", ifelse(grepl("AGBD", STABILITY$variable), "agbd_", "ht_"))
    series <- bind_rows(map_tiles(stab_tiles, function(t) cached(paste0(stab_key, t, "_n", sum(stab_plots$tile_num == t), "_y", length(STABILITY$years)),
                                                                 extract_series_tile(t)), "time series")) %>%
        filter(!is.na(value))

    per_plot <- series %>%
        arrange(uid, strategy, year) %>%
        group_by(uid, tile_num, strategy) %>%
        summarise(n_years = n(), sd = sd(value), mean_abs_step = mean(abs(diff(value))), mean = mean(value)) %>%
        ungroup() %>%
        filter(n_years >= 4)
    stab_pairs <- per_plot %>%
        select(uid, tile_num, strategy, sd, mean_abs_step) %>%
        pivot_wider(names_from = strategy, values_from = c(sd, mean_abs_step)) %>%
        filter(!is.na(sd_single), !is.na(sd_multi))
    stab_tests <- cluster_boot(stab_pairs %>% mutate(cluster = tile_num),
                               function(d) c(d_sd = median(d$sd_multi - d$sd_single),
                                             d_step = median(d$mean_abs_step_multi - d$mean_abs_step_single))) %>%
        mutate(better = "lower", winner = call_winner(lo, hi, better),
               single = c(median(stab_pairs$sd_single), median(stab_pairs$mean_abs_step_single)),
               multi  = c(median(stab_pairs$sd_multi),  median(stab_pairs$mean_abs_step_multi)))
    cat(sprintf("Mature undisturbed plots with >= 4 years in both series: %d (%d tiles). Variable: %s\n",
                nrow(stab_pairs), n_distinct(stab_pairs$tile_num), STABILITY$variable))
    print(stab_tests)
} else message("RUN_STABILITY = FALSE: section skipped")

In [ ]:
if (!is.null(stab_tests)) {
    options(repr.plot.width = 14, repr.plot.height = 5, repr.plot.res = 150)
    show_uid <- sample(unique(stab_pairs$uid), min(40, n_distinct(stab_pairs$uid)))
    traj <- series %>% filter(uid %in% show_uid) %>%
        group_by(uid, strategy) %>% mutate(anomaly = value - mean(value)) %>% ungroup() %>%
        mutate(strategy = factor(strategy, STRAT_LEVELS))
    pD1 <- ggplot(traj, aes(year, anomaly, group = uid, colour = strategy)) +
        geom_hline(yintercept = 0) +
        geom_line(alpha = 0.5) +
        facet_wrap(~ strategy, labeller = as_labeller(STRAT_LABELS)) +
        scale_strategy() + guides(colour = "none") +
        labs(x = NULL, y = paste("Departure from plot mean,", STABILITY$variable), title = "D1. Year-to-year variation in mature, undisturbed stands",
             subtitle = paste(length(show_uid), "random plots; flatter lines = less noise. Note:", STABILITY$caveat))
    pD2 <- ggplot(per_plot %>% mutate(strategy = factor(strategy, STRAT_LEVELS)), aes(strategy, sd, fill = strategy)) +
        geom_boxplot(outlier.shape = NA, width = 0.6) +
        coord_cartesian(ylim = c(0, quantile(per_plot$sd, 0.97))) +
        scale_x_discrete(labels = STRAT_LABELS) + scale_strategy() + guides(fill = "none") +
        labs(x = NULL, y = paste("SD across years,", STABILITY$variable), title = "D2. Per-plot noise")
    print(pD1 + pD2 + plot_layout(widths = c(2.2, 1)))
    s <- stab_tests %>% filter(metric == "d_sd")
    takeaway(sprintf("in %d mature undisturbed plots, the median year-to-year SD is %.2f (single) vs %.2f (multi); paired median difference %s (%s). Note: %s.",
                     nrow(stab_pairs), s$single, s$multi, fmt_ci(s$estimate, s$lo, s$hi), winner_phrase(s$winner), STABILITY$caveat))
}

## 6. Evidence E: effect on the boreal AGB stock
Small per-pixel differences add up over the biome: 1 Mg/ha over ~1.2 billion ha of boreal forest is ~1.2 Pg of biomass. Random pixel errors mostly cancel in a total; systematic differences do not.

- **6.1 Totals.** From each tile's `*boreal_eco.csv` (AGB in Pg per ecoregion and model iteration), keeping ecoregions of the boreal biome (`BOREAL_BIOME`). For each year, both runs are summed over the **same** tiles. The uncertainty is the model-iteration spread (treating tiles as independent). Where only some tiles have a single-year run, the ratio of the two totals on those tiles, with a tile-bootstrap CI, scales the difference to the full multi-year boreal total.
- **6.2 Where the difference comes from**, by continent and latitude band.
- **6.3 Which total the field plots favour.** For each map, the mean field-plot error is computed per *map* AGBD range (the right conditioning for adjusting a map total), and weighted by each range's share of the map's boreal pixels. This gives the relative bias each map's total would carry if the plot errors held everywhere. It is **indicative**: the plots are not a probability sample.

In [ ]:
run_id_of <- function(tindex_fn) c(version = str_match(tindex_fn, "BOREAL_MAP/([^/]+)/")[, 2],
                                   identifier = str_match(tindex_fn, "AGB_H30_\\d{4}/([^/]+)/AGB_tindex")[, 2])
stock_table <- if (file.exists(STOCK_TABLE_FN)) read_csv(STOCK_TABLE_FN, show_col_types = FALSE) else NULL

read_stock_tile <- function(year, t, strategy) {
    ti <- TINDEX[[strategy]][[as.character(year)]]
    files <- list_run_dir(ti$s3_path[ti$tile_num == t])
    f <- grep("boreal_eco\\.csv$", files, value = TRUE)[1]
    if (is.na(f)) stop("no *boreal_eco.csv for ", strategy)
    read_stats_csv(f) %>% transmute(ECO_ID = boreal_eco, iter, AGB_Pg)
}

# Per tile, iteration and run: AGB_Pg summed over boreal-biome ecoregions
stock_run <- function(year, strategy) {
    fn  <- if (strategy == "single") SINGLE_TINDEX[[as.character(year)]] else MULTI_TINDEX(year)
    rid <- run_id_of(fn)
    raw <- if (!is.null(stock_table) && any(stock_table$year == year & stock_table$IDENTIFIER == rid[["identifier"]] &
                                            stock_table$RELEASE_VERSION == rid[["version"]])) {
        message(sprintf("  stock %d %s: from %s", year, strategy, basename(STOCK_TABLE_FN)))
        stock_table %>% filter(year == !!year, IDENTIFIER == rid[["identifier"]], RELEASE_VERSION == rid[["version"]]) %>%
            transmute(tile_num, ECO_ID = boreal_eco, iter, AGB_Pg)
    } else {
        # every tile of the run: for the multi-year run this gives the boreal-wide total (one read per tile; cached)
        tiles <- TINDEX[[strategy]][[as.character(year)]]$tile_num
        bind_rows(map_tiles(tiles, function(t) cached(paste0("stock_", year, "_", strategy, "_", t),
                                                       read_stock_tile(year, t, strategy) %>% mutate(tile_num = t)),
                            sprintf("stock %d %s", year, strategy)))
    }
    raw %>% inner_join(eco_biome, by = "ECO_ID") %>% filter(BIOME == BOREAL_BIOME) %>%
        group_by(tile_num, iter) %>% summarise(AGB_Pg = sum(AGB_Pg)) %>% ungroup() %>%
        mutate(year = year, strategy = strategy)
}
stock_tile_iter <- bind_rows(lapply(YEARS, function(y) bind_rows(lapply(STRAT_LEVELS, function(s) stock_run(y, s)))))

stock_tile <- stock_tile_iter %>%
    group_by(year, strategy, tile_num) %>% summarise(Pg = mean(AGB_Pg), sd = sd(AGB_Pg)) %>% ungroup()
multi_domain <- stock_tile %>% filter(strategy == "multi") %>% group_by(year) %>% summarise(multi_domain_Pg = sum(Pg), multi_domain_tiles = n())
paired_tiles <- stock_tile %>% select(year, strategy, tile_num, Pg, sd) %>%
    pivot_wider(names_from = strategy, values_from = c(Pg, sd)) %>%
    filter(!is.na(Pg_single), !is.na(Pg_multi)) %>%
    left_join(tile_info, by = "tile_num")

stock_tests <- paired_tiles %>% group_by(year) %>%
    group_modify(function(d, key) {
        b <- cluster_boot(d %>% mutate(cluster = tile_num), function(x) c(ratio = sum(x$Pg_multi) / sum(x$Pg_single)))
        tibble(tiles = nrow(d), single_Pg = sum(d$Pg_single), multi_Pg = sum(d$Pg_multi),
               diff_Pg = sum(d$Pg_multi) - sum(d$Pg_single),
               diff_iter_ci = 1.96 * sqrt(sum(d$sd_single^2, na.rm = TRUE) + sum(d$sd_multi^2, na.rm = TRUE)),
               ratio = b$estimate, ratio_lo = b$lo, ratio_hi = b$hi)
    }) %>% ungroup() %>%
    left_join(multi_domain, by = "year") %>%
    mutate(pct_diff = 100 * (ratio - 1),
           full_domain = tiles == multi_domain_tiles,
           # difference scaled to the full boreal total: multi_domain * (1 - 1/ratio)
           domain_diff_Pg = ifelse(full_domain, diff_Pg, multi_domain_Pg * (1 - 1 / ratio)),
           domain_diff_lo = ifelse(full_domain, diff_Pg - diff_iter_ci, multi_domain_Pg * (1 - 1 / ratio_lo)),
           domain_diff_hi = ifelse(full_domain, diff_Pg + diff_iter_ci, multi_domain_Pg * (1 - 1 / ratio_hi)))
stock_tests %>% transmute(year, tiles, full_domain, single_Pg = round(single_Pg, 2), multi_Pg = round(multi_Pg, 2),
                          `multi - single on these tiles (Pg)` = fmt_ci(diff_Pg, diff_Pg - diff_iter_ci, diff_Pg + diff_iter_ci),
                          `multi / single - 1 (%)` = fmt_ci(pct_diff, 100 * (ratio_lo - 1), 100 * (ratio_hi - 1), 1),
                          `boreal-wide difference (Pg)` = fmt_ci(domain_diff_Pg, domain_diff_lo, domain_diff_hi))

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 5, repr.plot.res = 150)
contrib <- paired_tiles %>%
    filter(!is.na(lat_band)) %>%
    group_by(year, continent, lat_band) %>%
    summarise(diff_Pg = sum(Pg_multi - Pg_single), single_Pg = sum(Pg_single), tiles = n()) %>% ungroup()
contrib <- contrib %>% mutate(pct = 100 * diff_Pg / single_Pg)
pE1 <- ggplot(contrib, aes(lat_band, pct, fill = pct > 0)) +
    geom_col(width = 0.7) +
    geom_hline(yintercept = 0) +
    geom_text(aes(label = sprintf("%+.2f Pg", diff_Pg), vjust = ifelse(pct > 0, -0.3, 1.2)), size = 2.6) +
    facet_grid(year ~ continent) +
    scale_y_continuous(expand = expansion(mult = 0.2)) +
    scale_fill_manual(values = c("TRUE" = STRAT_COLORS[["multi"]], "FALSE" = STRAT_COLORS[["single"]]), guide = "none") +
    labs(x = NULL, y = "Multi - single (% of single-year stock)", title = "E1. Where the stock difference comes from",
         subtitle = "Boreal-biome AGB on the tiles mapped by both runs; labels: difference in Pg on those tiles") +
    theme(axis.text.x = element_text(angle = 45, hjust = 1))

# 6.3 Field-plot error by MAP AGBD range, weighted by the map's own area in each range
plot_bias <- bind_rows(lapply(STRAT_LEVELS, function(s) boreal_pairs %>%
    transmute(strategy = s, bin = agbd_bin(.data[[paste0("pred_", s)]]), err = .data[[paste0("err_", s)]]))) %>%
    group_by(strategy, bin) %>% summarise(n_plots = n(), bias = ifelse(n() >= MIN_PLOTS_BIN, mean(err), NA_real_)) %>% ungroup()
area_share <- bind_rows(lapply(STRAT_LEVELS, function(s) pix_b %>%
    transmute(year, strategy = s, value = .data[[paste0("agb_", s)]], bin = agbd_bin(.data[[paste0("agb_", s)]])))) %>%
    group_by(year, strategy) %>% mutate(map_mean = mean(value)) %>%
    group_by(year, strategy, bin, map_mean) %>% summarise(share = n()) %>%
    group_by(year, strategy) %>% mutate(share = share / sum(share)) %>% ungroup()
implied <- area_share %>% left_join(plot_bias, by = c("strategy", "bin")) %>%
    group_by(year, strategy) %>%
    summarise(covered_share = sum(share[!is.na(bias)]),
              rel_bias = sum(share * bias, na.rm = TRUE) / sum(share[!is.na(bias)]) / first(map_mean)) %>% ungroup() %>%
    left_join(stock_tests %>% select(year, single_Pg, multi_Pg, tiles, full_domain, multi_domain_Pg, ratio), by = "year") %>%
    # boreal-wide totals: as summed when the year is full-domain, otherwise scaled from the paired tiles
    mutate(total_Pg = case_when(full_domain & strategy == "single" ~ single_Pg, full_domain ~ multi_Pg,
                                strategy == "single" ~ multi_domain_Pg / ratio, TRUE ~ multi_domain_Pg),
           implied_error_Pg = total_Pg * rel_bias)
pE2 <- ggplot(plot_bias, aes(bin, bias, colour = strategy, group = strategy)) +
    geom_hline(yintercept = 0) +
    geom_line(linewidth = 0.9) + geom_point(aes(size = n_plots)) +
    scale_strategy() + scale_size_area(max_size = 4, name = "plots") +
    labs(x = "Map AGBD (Mg/ha)", y = "Mean map - field (Mg/ha)", title = "E2. Field-plot error by map AGBD range",
         subtitle = sprintf("Ranges with < %d plots are omitted", MIN_PLOTS_BIN))
print(pE1 + pE2 + plot_layout(widths = c(1.6, 1)))
implied %>% transmute(year, strategy, `boreal-wide total (Pg)` = round(total_Pg, 2), full_domain, `area covered by plot bins` = percent(covered_share, 1),
                      `implied relative bias` = percent(rel_bias, 0.1), `implied error in total (Pg)` = round(implied_error_Pg, 2))

In [ ]:
st_y <- stock_tests %>% arrange(desc(tiles)) %>% slice(1)          # the year with the most paired tiles (2020: full domain)
imp_y <- implied %>% filter(year == st_y$year)
takeaway(sprintf("in %d, switching to the multi-year model changes the boreal-biome AGB stock by %s Pg (%s%%)%s. If the field-plot errors held everywhere, the single-year total would be off by %+.2f Pg and the multi-year total by %+.2f Pg (indicative).",
                 st_y$year, fmt_ci(st_y$domain_diff_Pg, st_y$domain_diff_lo, st_y$domain_diff_hi),
                 fmt_ci(st_y$pct_diff, 100 * (st_y$ratio_lo - 1), 100 * (st_y$ratio_hi - 1), 1),
                 ifelse(st_y$full_domain, sprintf(" on %d tiles (single %.2f vs multi %.2f Pg)", st_y$tiles, st_y$single_Pg, st_y$multi_Pg),
                        sprintf(", scaled from %d paired tiles", st_y$tiles)),
                 imp_y$implied_error_Pg[imp_y$strategy == "single"], imp_y$implied_error_Pg[imp_y$strategy == "multi"]))

**Reading E.** The stock table gives, per year, both totals on the same tiles and their difference. E1 shows which regions carry the difference. E2 and the implied-error table answer "which total is more plausible": a map whose plot errors are near zero across the map AGBD ranges that hold most of its area carries little implied error in its total.

## 7. Scorecard and verdict
One row per criterion: each strategy's value, the paired difference (multi − single) with its 95 % CI, and the winner (CI must exclude zero). **Role**: field-plot accuracy (§4) decides; training data (§2) and stability (§5) support or qualify; map differences (§3) are descriptive; the stock rows (§6) give the size of the effect.

In [ ]:
acc_main <- acc_tests %>% filter(domain == HEADLINE)
score <- bind_rows(
    acc_main %>% filter(metric %in% c("d_rmsd", "d_absbias", "pct_multi_closer_minus50")) %>%
        transmute(evidence = "C. Accuracy vs field plots", role = "primary",
                  criterion = recode(metric, d_rmsd = "RMSD, all years (Mg/ha)", d_absbias = "|Bias|, all years (Mg/ha)",
                                     pct_multi_closer_minus50 = "Plot-years where multi-year is closer (% - 50)"),
                  single = c(d_rmsd = acc_abs$estimate[acc_abs$metric == "rmsd_single"],
                             d_absbias = abs(acc_abs$estimate[acc_abs$metric == "bias_single"]),
                             pct_multi_closer_minus50 = NA)[metric],
                  multi  = c(d_rmsd = acc_abs$estimate[acc_abs$metric == "rmsd_multi"],
                             d_absbias = abs(acc_abs$estimate[acc_abs$metric == "bias_multi"]),
                             pct_multi_closer_minus50 = NA)[metric],
                  estimate, lo, hi, better, winner),
    if (length(YEARS) > 1) acc_tests %>% filter(metric == "d_rmsd", str_detect(domain, "^Year")) %>%
        mutate(yr = as.integer(sub("Year ", "", domain))) %>%
        left_join(acc_abs_year %>% filter(metric %in% c("rmsd_single", "rmsd_multi")) %>%
                      select(yr = year, m = metric, v = estimate) %>% pivot_wider(names_from = m, values_from = v), by = "yr") %>%
        transmute(evidence = "C. Accuracy vs field plots", role = "primary", criterion = paste("RMSD,", yr, "(Mg/ha)"),
                  single = rmsd_single, multi = rmsd_multi, estimate, lo, hi, better, winner),
    train_tests %>% filter(year == "all") %>%
        transmute(evidence = "A. Training data", role = ifelse(is.na(better), "descriptive", "supporting"),
                  criterion = ifelse(log2ratio, paste(label, "(log2 ratio)"), label),
                  single, multi, estimate, lo, hi, better, winner),
    map_tests %>% transmute(evidence = "B. Map differences", role = "descriptive",
                            criterion = recode(metric, mean_diff = "Mean AGBD difference, boreal (Mg/ha)", sd_diff = "Mean map uncertainty sd (Mg/ha; niter may differ)"),
                            single = NA_real_, multi = NA_real_, estimate, lo, hi, better = NA_character_, winner = "descriptive"),
    if (!is.null(stab_tests)) stab_tests %>%
        transmute(evidence = "D. Temporal stability", role = "supporting",
                  criterion = recode(metric, d_sd = paste("Median year-to-year SD,", STABILITY$variable),
                                     d_step = paste("Median abs. annual step,", STABILITY$variable)),
                  single, multi, estimate, lo, hi, better, winner),
    stock_tests %>% transmute(evidence = "E. Boreal AGB stock", role = "effect",
                              criterion = paste0("Boreal AGB stock, ", year, ifelse(full_domain, "", " (scaled from sampled tiles)"), " (Pg)"),
                              single = ifelse(full_domain, single_Pg, NA), multi = ifelse(full_domain, multi_Pg, NA),
                              estimate = domain_diff_Pg, lo = domain_diff_lo, hi = domain_diff_hi, better = NA_character_, winner = "descriptive"),
    implied %>% select(year, strategy, implied_error_Pg) %>% pivot_wider(names_from = strategy, values_from = implied_error_Pg) %>%
        transmute(evidence = "E. Boreal AGB stock", role = "effect", criterion = paste0("Error in total implied by plot biases, ", year, " (Pg)"),
                  single, multi, estimate = abs(multi) - abs(single), lo = NA_real_, hi = NA_real_, better = NA_character_, winner = "descriptive")
) %>%
    mutate(difference = fmt_ci(estimate, lo, hi),
           role = factor(role, c("primary", "supporting", "effect", "descriptive")))

score %>% arrange(role, evidence) %>%
    transmute(role, evidence, criterion, single = round(single, 2), multi = round(multi, 2),
              `multi - single [95% CI]` = difference, `better if` = better, winner)

In [ ]:
prim <- score %>% filter(role == "primary", criterion == "RMSD, all years (Mg/ha)")
supp <- score %>% filter(role == "supporting", winner %in% STRAT_LEVELS) %>% count(winner)
supp_txt <- if (nrow(supp)) paste(sprintf("%d supporting criteria favour %s-year", supp$n, supp$winner), collapse = "; ") else "no supporting criterion shows a clear difference"
st_txt <- sprintf("Switching to multi-year changes the %d boreal AGB stock by %s Pg (%+.1f%%).", st_y$year,
                  fmt_ci(st_y$domain_diff_Pg, st_y$domain_diff_lo, st_y$domain_diff_hi), st_y$pct_diff)

verdict <- if (prim$winner %in% STRAT_LEVELS) {
    sprintf("VERDICT: the %s-year model is better. It is more accurate against independent field plots (RMSD %.1f vs %.1f Mg/ha over %s; difference %s). %s. %s",
            prim$winner, prim[[prim$winner]], prim[[setdiff(STRAT_LEVELS, prim$winner)]], paste(YEARS, collapse = ", "), prim$difference, supp_txt, st_txt)
} else if (nrow(supp) == 1) {
    sprintf("VERDICT: field plots cannot separate the two maps (RMSD difference %s Mg/ha). The case for the %s-year model rests on supporting evidence: %s. %s",
            prim$difference, supp$winner, supp_txt, st_txt)
} else {
    sprintf("VERDICT: inconclusive. Field plots cannot separate the maps (RMSD difference %s Mg/ha), and %s. %s", prim$difference, supp_txt, st_txt)
}
cat(strwrap(verdict, 110), sep = "\n")
cat("\nConfounds to state with this result:", paste0("\n - ", c(CONFOUNDS, if (!is.null(stab_tests)) paste("stability:", STABILITY$caveat))), "\n")

In [ ]:
options(repr.plot.width = 13, repr.plot.height = 0.42 * nrow(score) + 2, repr.plot.res = 150)
card <- score %>%
    mutate(row = factor(paste0(evidence, ": ", criterion), rev(paste0(evidence, ": ", criterion))),
           winner_lab = ifelse(winner %in% STRAT_LEVELS, paste0(winner, "-year"), winner))
pS <- ggplot(card, aes(y = row)) +
    geom_tile(aes(x = 1, fill = winner), colour = "white", width = 0.95) +
    geom_text(aes(x = 1, label = winner_lab), size = 3.3) +
    geom_text(aes(x = 2, label = ifelse(is.na(single), "", formatC(single, format = "f", digits = 2))), size = 3.3) +
    geom_text(aes(x = 3, label = ifelse(is.na(multi), "", formatC(multi, format = "f", digits = 2))), size = 3.3) +
    geom_text(aes(x = 4.6, label = difference), size = 3.3) +
    geom_text(aes(x = 5.9, label = as.character(role)), size = 3.3, colour = "grey35") +
    scale_x_continuous(breaks = c(1, 2, 3, 4.6, 5.9), labels = c("Winner", "Single", "Multi", "Multi - single [95% CI]", "Role"),
                       position = "top", limits = c(0.5, 6.5)) +
    scale_fill_manual(values = WINNER_COLORS, guide = "none") +
    labs(x = NULL, y = NULL, title = "Scorecard: single-year vs multi-year ICESat-2 training",
         caption = paste(strwrap(verdict, 120), collapse = "\n")) +
    theme_minimal(base_size = 11) +
    theme(panel.grid = element_blank(), axis.text.x.top = element_text(face = "bold"),
          plot.title = element_text(face = "bold"), plot.caption = element_text(hjust = 0, size = 10),
          plot.title.position = "plot", plot.caption.position = "plot")
pS

### Export
The scorecard, verdict, per-tile and per-plot tables, and the main figures go to `OUT_DIR`.

In [ ]:
write_csv(score %>% mutate(role = as.character(role)), file.path(OUT_DIR, "scorecard.csv"))
writeLines(c(verdict, "", "Confounds:", CONFOUNDS, if (!is.null(stab_tests)) paste("stability:", STABILITY$caveat)),
           file.path(OUT_DIR, "verdict.txt"))
write_csv(train_sum, file.path(OUT_DIR, "A_training_by_tile_year.csv"))
write_csv(year_counts, file.path(OUT_DIR, "A_obs_by_icesat2_year.csv"))
write_csv(tile_diff, file.path(OUT_DIR, "B_map_difference_by_tile_year.csv"))
write_csv(by_bin, file.path(OUT_DIR, "B_difference_by_agbd_range.csv"))
write_csv(pairs, file.path(OUT_DIR, "C_field_plot_pairs.csv"))
write_csv(acc_tests, file.path(OUT_DIR, "C_accuracy_tests.csv"))
write_csv(net_tab, file.path(OUT_DIR, "C_direction_of_change_by_range.csv"))
if (!is.null(stab_tests)) write_csv(stab_pairs, file.path(OUT_DIR, "D_stability_by_plot.csv"))
write_csv(stock_tests, file.path(OUT_DIR, "E_stock_by_year.csv"))
write_csv(implied, file.path(OUT_DIR, "E_stock_implied_by_plot_bias.csv"))
save_fig <- function(p, name, w, h) ggsave(file.path(OUT_DIR, paste0(name, ".png")), p, width = w, height = h, dpi = 200, bg = "white")
save_fig(pS, "S_scorecard", 13, 0.42 * nrow(score) + 2)
save_fig(pA0, "A0_obs_by_icesat2_year", 14, 5)
save_fig(pA1, "A1_training_paired", 14, 4.5)
save_fig(pB1 + pB2 + pB3 + plot_layout(widths = c(1, 1.3, 1.1)), "B1_map_agreement", 15, 5)
save_fig(pB4a + pB4b, "B4_difference_by_agbd_range", 15, 5)
save_fig(pC1 + pC2 + plot_layout(widths = c(1.4, 1)), "C1_accuracy", 15, 5.5)
save_fig(pC3 + pC4 + plot_layout(widths = c(1, 1.5)), "C3_error_and_direction_by_range", 15, 5)
save_fig(pE1 + pE2 + plot_layout(widths = c(1.6, 1)), "E_stock", 15, 5)
message("Outputs in ", OUT_DIR)

## Notes and limits
- **More years.** Add each new single-year run to `SINGLE_TINDEX`; every section then includes that year, and §5 switches to AGBD series once four years exist. Cached per-tile results are reused.
- **Field plots are old and few in some regions.** The paired design removes their shared error, but regions with few plot groups get wide CIs (C2). LVIS (ABoVE domain, 2017 and 2019) is the next independent reference for a same-year comparison; the extraction code is in `validation_v3.1_lidar_LVIS_GLiHT.ipynb`.
- **Stock uncertainty** here is the model-iteration spread, with tiles treated as independent; it understates uncertainty if iteration errors are correlated across tiles. The implied-error estimate (§6.3) depends on how representative the plots are of each AGBD range.
- **Confounds** listed in §0 are printed with the verdict. If the runs used different `mapBoreal` commits, part of any difference is due to code, not training data.
- **Stability ≠ accuracy.** A smoother series is better only if real change is still captured; see the LVIS change test before claiming the multi-year series is better for trends.